# Qwen2.5-7B CodeContest Eval on TPU

Loads Qwen2.5-7B-Instruct via the tunix JAX stack, generates Python solutions
for CodeContest problems, and evaluates them by running the code against the
provided test cases.

Code execution uses threads (no `fork`, no `subprocess`) so it is safe to run
after JAX/TPU initialisation.

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
MODEL_PATH = "/path/to/qwen2.5-7b-instruct"  # local safetensors directory
DATA_PATH  = "/path/to/codecontest.json"      # JSON list of problem dicts

# ── TPU topology ───────────────────────────────────────────────────────────
TP_SIZE = 8   # tensor-parallel chips; use 1 for a single chip

# ── generation ─────────────────────────────────────────────────────────────
K_CODE         = 4      # code samples per problem (pass@K)
MAX_NEW_TOKENS = 1024
MAX_CACHE_LEN  = 8192   # KV-cache slots; must be >= max_prompt_len + MAX_NEW_TOKENS
TEMPERATURE    = 0.8
TOP_P          = 0.95

# ── evaluation ─────────────────────────────────────────────────────────────
MAX_TEST     = 5    # max test cases to check per problem
EXEC_TIMEOUT = 5.0  # seconds per test case

In [ ]:
import io
import json
import re
import sys
import threading
import typing

import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
from jax.sharding import Mesh
from transformers import AutoTokenizer

from tunix.generate import sampler as sampler_lib
from tunix.models.qwen2 import model as model_lib
from tunix.models.qwen2 import params as params_lib

print(f"JAX backend : {jax.default_backend()}")
print(f"JAX devices : {jax.devices()}")

## Load model

In [ ]:
devices      = jax.devices()
mesh_devices = np.array(devices[:TP_SIZE]).reshape(1, TP_SIZE)
mesh         = Mesh(mesh_devices, axis_names=("fsdp", "tp"))

tokenizer    = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

model_config = model_lib.ModelConfig.qwen2p5_7b_instruct()
model        = params_lib.create_model_from_safe_tensors(
    file_dir=MODEL_PATH,
    config=model_config,
    mesh=mesh,
    dtype=jnp.bfloat16,
)

sampler = sampler_lib.Sampler(
    model,
    tokenizer,
    sampler_lib.CacheConfig(
        cache_size=MAX_CACHE_LEN,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)
print("Model and sampler ready.")

## Load data

In [ ]:
# Expected fields per problem dict:
#   question        : str
#   test_input      : list[str]
#   test_output     : list[str]
#   test_time_limit : float  (optional, defaults to 2.0)
with open(DATA_PATH) as f:
    data = json.load(f)

print(f"Loaded {len(data)} problems.")

## Prompt builder + code extractor

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert competitive programmer. "
    "Solve the problem below in Python. "
    "Output ONLY a single ```python``` code block containing a complete, "
    "runnable solution that reads from stdin and writes to stdout."
)


def build_prompt(problem: str) -> str:
    """Apply the Qwen chat template so the sampler receives a ready-to-tokenise string."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def extract_code(text: str) -> str | None:
    matches = re.findall(r"```python(.*?)```", text, re.DOTALL)
    return matches[-1].strip() if matches else None

## Code execution (thread-based, TPU-safe)

We run generated code in a daemon thread rather than a subprocess to avoid
forking the process after JAX/TPU initialisation. The thread shares the
interpreter but runs in an isolated `exec` context with its own `input`/stdout.

In [ ]:
import ctypes
import io
import sys
import threading
import typing

_exec_lock = threading.Lock()


def _kill_thread(t: threading.Thread) -> None:
    """Inject SystemExit into a running thread to stop infinite loops."""
    if t.ident is None:
        return
    ctypes.pythonapi.PyThreadState_SetAsyncExc(
        ctypes.c_ulong(t.ident),
        ctypes.py_object(SystemExit),
    )


def run_code(code: str, stdin_str: str, timeout: float = EXEC_TIMEOUT) -> str:
    """Execute *code* with *stdin_str* as stdin.

    stdout/stdin are redirected and restored in the main thread's finally block
    so they are always restored even when the child thread times out.
    Timed-out threads are killed via ctypes so they don't linger and thrash the GIL.
    """
    result: list[str | None] = [None]
    buf = io.StringIO()

    def target() -> None:
        lines_iter = iter(stdin_str.splitlines())

        def fake_input(prompt: str = "") -> str:
            try:
                return next(lines_iter)
            except StopIteration:
                raise EOFError

        ctx: dict = {
            "__name__": "__main__",
            "input": fake_input,
            "List":     typing.List,
            "Tuple":    typing.Tuple,
            "Optional": typing.Optional,
        }
        try:
            exec(compile(code, "<solution>", "exec"), ctx)  # noqa: S102
            result[0] = buf.getvalue()
        except SystemExit:
            result[0] = buf.getvalue()
        except Exception as exc:
            result[0] = f"error: {exc}"

    with _exec_lock:
        saved_stdout, saved_stdin = sys.stdout, sys.stdin
        sys.stdout = buf
        sys.stdin  = io.StringIO(stdin_str)
        try:
            t = threading.Thread(target=target, daemon=True)
            t.start()
            t.join(timeout=timeout)
        finally:
            sys.stdout = saved_stdout
            sys.stdin  = saved_stdin

    if t.is_alive():
        _kill_thread(t)
        return "timeout"
    return result[0] if result[0] is not None else "error: no output"


def outputs_match(actual: str, expected: str) -> bool:
    return " ".join(actual.split()) == " ".join(expected.split())


def evaluate_code(
    code: str,
    test_inputs: list[str],
    test_outputs: list[str],
    time_limit: float,
) -> dict:
    n = min(len(test_inputs), MAX_TEST)
    passed = sum(
        outputs_match(run_code(code, inp, timeout=time_limit), exp)
        for inp, exp in zip(test_inputs[:n], test_outputs[:n])
    )
    return {"passed": passed, "total": n}

## Smoke-test: code execution harness

Run this cell to verify `run_code` / `evaluate_code` work correctly **before**
loading the model. Tests four scenarios: correct code, wrong output, runtime
error, and infinite loop (timeout).

In [ ]:
import random

# ── 20 test cases for "read a and b, print a+b" ───────────────────────────
rng = random.Random(42)
_pairs         = [(rng.randint(-1000, 1000), rng.randint(-1000, 1000)) for _ in range(20)]
_smoke_inputs  = [f"{a} {b}" for a, b in _pairs]
_smoke_outputs = [str(a + b)  for a, b in _pairs]

# ── four candidate solutions ───────────────────────────────────────────────
_cases = {
    "correct": ("""\
a, b = map(int, input().split())
print(a + b)
""", _smoke_inputs, _smoke_outputs),
    "wrong output": ("""\
a, b = map(int, input().split())
print(a - b)
""", _smoke_inputs, _smoke_outputs),
    "runtime error": ("""\
a, b = map(int, input().split())
print(1 / 0)
""", _smoke_inputs, _smoke_outputs),
    # Only 1 timeout case — each waits the full timeout, no need to repeat 20x
    "timeout": ("""\
a, b = map(int, input().split())
while True:
    pass
""", _smoke_inputs[:1], _smoke_outputs[:1]),
}

# ── run and report ─────────────────────────────────────────────────────────
print(f"{'case':<16} {'passed':>6}  {'total':>5}  per-test results")
print("-" * 68)

all_ok = True
for label, (code, inputs, outputs) in _cases.items():
    per_test = []
    for inp, exp in zip(inputs, outputs):
        out = run_code(code.strip(), inp, timeout=1.0)
        per_test.append("P" if outputs_match(out, exp) else "F")
    passed = per_test.count("P")
    total  = len(per_test)
    print(f"{label:<16} {passed:>6}  {total:>5}  {''.join(per_test)}")

    if label == "correct"      and passed != total: all_ok = False
    if label == "wrong output" and passed == total: all_ok = False
    if label == "timeout"      and passed == total: all_ok = False

print()
print("Harness OK — safe to proceed." if all_ok else "HARNESS ISSUE — check run_code / outputs_match.")

## Generate and evaluate

In [ ]:
PROBLEMS = data[:10]  # adjust slice as needed

all_results = []

for prob_idx, prob in enumerate(PROBLEMS):
    question    = prob["question"]
    test_inputs = prob["test_input"]
    test_outputs= prob["test_output"]
    time_limit  = float(prob.get("test_time_limit", 2.0))

    # Send K_CODE copies of the same prompt in a single batched call.
    prompt  = build_prompt(question)
    out     = sampler(
        input_strings=[prompt] * K_CODE,
        max_generation_steps=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )

    # Pick the best-scoring sample (most test cases passed).
    best = {"passed": 0, "total": 0, "code": None}
    for gen_text in out.text:
        code = extract_code(gen_text)
        if code is None:
            continue
        score = evaluate_code(code, test_inputs, test_outputs, time_limit)
        if score["passed"] > best["passed"]:
            best = {**score, "code": code}

    all_results.append(best)
    solved = best["total"] > 0 and best["passed"] == best["total"]
    marker = "PASS" if solved else "FAIL"
    print(f"[{prob_idx+1:3d}/{len(PROBLEMS)}] {marker}  {best['passed']}/{best['total']} tests")

## Summary

In [ ]:
solved = sum(
    1 for r in all_results if r["total"] > 0 and r["passed"] == r["total"]
)
total = len(all_results)
print(f"pass@{K_CODE}  {solved}/{total}  ({solved/total:.1%})")